# Connected two-fin: eigenpolarization transport and complex-frequency characterization

**The scientific question:** Given the optimized, *single-layer* connected two-fin geometry, can we establish robust control of nonorthogonal eigenpolarizations $\eta$ over a physically accessible parameter; and does the resulting eigenvector path imply any **physical** ordered polarization transport in one transverse metasurface layer?

This notebook uses the **same** `engine_twofin_eta.py`, six bounded geometric parameters, material/height settings, and saved optimizer checkpoint. It does **not** optimize a multilayer or switch back to Fourier contours.

We deliberately separate three distinct objects:

1. $T_{21}(\lambda,p,\theta)$: experimentally accessible **far-field** forward Jones transmission of the periodic unit cell. $\eta=|\langle R_+|R_-\rangle|$ describes its *right* eigenvectors.
2. An ordered **kinematic** sequence of Jones eigenvectors in geometry/orientation parameter space; its biorthogonal Wilson product is defined mathematically, but **does not automatically occur as light passes through an array of different pixels**.
3. Actual Maxwell fields and full scattering of periodic/supercell/finite structures; the only legitimate source of claims about physical propagation, reflection, resonances, and energy transport.

**Execution:** run cells in order; expensive complex-frequency, depth-field and array experiments are opt-in. All geometries, angles and wavelengths are in µm/degrees unless indicated.

## 1 · Checkpoint, physical configuration and independent runtime switches

The revised optimizer writes `results/twofin_eta_transport/twofin_eta_checkpoint_v2.npz` (or `twofin_eta_final.npz`). The actual optimizer-saved material indices, period, thickness, modulation, bounds and parameters are loaded. A missing or incompatible checkpoint raises an error; there is **no silent default geometry**.

In [ ]:
from pathlib import Path
from dataclasses import replace, asdict
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
from engine_twofin_eta import (Physics, RCWA, Modulation, PARAM_KEYS, BOUNDS, vary,
    twofin_mask, solve_torch, check_connected, export_layout, diffraction_audit,
    circular_conversion, supercell_solve, device_dtype)
from engine_twofin_characterization import (
    read_design, polarization_metrics, tracked_modes, wilson_cycle,
    rectangular_parameter_loop, real_frequency_s, solve_complex_s,
    rational_fit, depth_response)

OUT = Path('results/twofin_eta_characterization')
OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = Path('results/twofin_eta_transport')  # optimizer output
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN_ORDER_CONVERGENCE = True
RUN_RASTER_CONVERGENCE = False
RUN_MODULATION_SWEEP = True
RUN_PARAMETER_SENSITIVITY = True
RUN_WILSON = True
RUN_WILSON_REFINEMENT = False
RUN_PB_LIBRARY = True
RUN_REAL_SPECTRUM = True
RUN_COMPLEX_FREQUENCY = False   # computationally expensive / exploratory
RUN_Z_FIELDS = False            # independent full-field Maxwell check
RUN_SUPERCELL_ORDER_TEST = False
RUN_FINITE_ARRAY = False

# Every optical measurement uses a hard binary geometry unless otherwise stated.
BASE = RCWA(order=8,raster=256,precision='complex128',device=DEVICE,edge_um=.003)
VERIFY_ORDERS = (6,8,10,12)
VERIFY_RASTERS = (192,256,320)

saved = read_design(CHECKPOINT_DIR)
PHYS, MOD = saved['physics'], saved['modulation']
RAW, GEOM = saved['raw'], saved['parameters']
p = {k:torch.tensor(v,device=DEVICE,dtype=torch.float64) for k,v in GEOM.items()}
print('Checkpoint:',saved['path'])
print('Objective tag:',saved['tag'])
print('Physical system:',asdict(PHYS))
print('Modulation:',asdict(MOD))
print('Geometry:',{k:round(v,6) for k,v in GEOM.items()})
print('Device:',DEVICE)

## 2 · Geometry, diffraction channels and far-field meaning of η

For this periodic cell, higher Fourier harmonics (including evanescent exterior orders) are retained and coupled by RCWA. **η is the overlap of the two eigenvectors of the *zero-order far-field* $T_{21}$**, not a local microscopic eigenpolarization. Above the exterior Rayleigh threshold, the specular $4\times4$ block is no longer the complete propagating scattering matrix.

In [ ]:
overlap_area,connected = check_connected(GEOM)
if not connected: raise RuntimeError('The loaded two fins are not physically connected.')
fig,axs=plt.subplots(1,3,figsize=(12,3.8))
for ax,u in zip(axs,[-1.,0.,1.]):
    q=vary(p,MOD,u)
    with torch.no_grad(): fill=twofin_mask(q,PHYS,256,hard=True).detach().cpu().numpy()
    half=PHYS.period/2
    ax.imshow(fill.T,origin='lower',extent=(-half,half,-half,half),
              cmap='Blues',interpolation='nearest',vmin=0,vmax=1)
    ax.set(aspect='equal',xlabel='x (µm)',ylabel='y (µm)',title=f'u={u:+.1f} · {MOD.parameter}={float(q[MOD.parameter]):.2f}')
plt.tight_layout();plt.show()
print(f'Connected overlap: {overlap_area*1e6:.1f} nm²')
print('Union layout export:',export_layout(GEOM,PHYS,OUT/'connected_twofin'))
print('Exterior diffraction check:')
for side in ('bottom','top'):
    row=diffraction_audit(PHYS)[side]
    print(side, row)
print('Zero-order exterior is the only propagating spatial channel:',
      diffraction_audit(PHYS)['exterior_only_specular'])

## 3 · Full complex scattering matrix, right/left eigenpolarizations, physical consistency

We inspect $S=\begin{pmatrix}R_{11}&T_{12}\\T_{21}&R_{22}\end{pmatrix}$ in the fixed laboratory $(x,y)$ basis and power-normalized propagating channels. Do not infer loss from $T_{21}$ alone; reflection is significant in a non-normal yet lossless transmission design. A scalar or nearly defective Jones matrix can have numerically meaningless eigenpolarizations. The helper checks finite non-scalar strength and gap.

In [ ]:
t0=time.perf_counter()
S0=real_frequency_s(PHYS,BASE,p)
M0=polarization_metrics(S0)
print(f'Baseline solved in {time.perf_counter()-t0:.1f} s')
print('S, channel order [bottom x, bottom y, top x, top y]:')
print(np.array2string(S0,precision=5,suppress_small=True))
for k in ('eta','gap','anisotropy','nonnormality','condition_number',
          'transmitted_x','transmitted_y','reflected_x','reflected_y',
          'unitarity_error','reciprocity_error','defined'):
    print(f'{k:>20s}: {M0[k]}')
print('T21 eigenvalues:',M0['eigenvalues'])
print('Right eigenvectors (columns):\n',M0['eigenvectors'])
# Verify direct eigenvectors against the independently implemented discriminant formula.
from engine_twofin_eta import jones_metrics
Ttorch=torch.as_tensor(S0[2:,:2],dtype=torch.complex128)
eta_algebra=float(jones_metrics(Ttorch)['eta'])
print('η from eigenvectors:',M0['eta'],' | analytic identity:',eta_algebra)
assert abs(M0['eta']-eta_algebra)<1e-7
if not M0['defined']:print('WARNING: eigenvectors are not numerically admissible at this point.')

## 4 · Independent Fourier-order and mask-raster convergence

Previous optimizations displayed sizable order dependence in $\eta$ and $d\eta/dp$. A reliable design must converge **Jones entries and eigenvector overlap**. Changing Fourier order and raster simultaneously confounds the diagnosis. The optional second cell varies raster at fixed Fourier order.

In [ ]:
order_results=[]
if RUN_ORDER_CONVERGENCE:
    for order in VERIFY_ORDERS:
        settings=replace(BASE,order=order)
        data=[]
        for u in (-1.,0.,1.):
            S=real_frequency_s(PHYS,settings,vary(p,MOD,u))
            data.append((S,polarization_metrics(S)))
        order_results.append(dict(order=order,data=data))
        print(f"order={order:2d} η[-,0,+]={[round(x[1]['eta'],4) for x in data]} "
              f"g_min={min(x[1]['gap'] for x in data):.4f} "
              f"Δη={max(x[1]['eta'] for x in data)-min(x[1]['eta'] for x in data):.4f} "
              f"unitarity_max={max(x[1]['unitarity_error'] for x in data):.2e}")
    fig,axs=plt.subplots(1,2,figsize=(11,3.5))
    for iu,u in enumerate((-1.,0.,1.)):
        axs[0].plot([r['order'] for r in order_results],
                    [r['data'][iu][1]['eta'] for r in order_results],'-o',label=f'u={u:+g}')
    axs[0].set(xlabel='Fourier order',ylabel='η',title='Eigenvector overlap')
    for iu,u in enumerate((-1.,0.,1.)):
        errs=[np.linalg.norm(r['data'][iu][0]-order_results[-1]['data'][iu][0],'fro') for r in order_results]
        axs[1].semilogy([r['order'] for r in order_results],np.maximum(errs,1e-16),'-o',label=f'u={u:+g}')
    axs[1].set(xlabel='Fourier order',ylabel='||S - S_highest||F',title='Complex full-S convergence')
    for ax in axs:ax.legend();ax.grid(alpha=.2)
    plt.tight_layout();plt.show()

In [ ]:
raster_results=[]
if RUN_RASTER_CONVERGENCE:
    order=VERIFY_ORDERS[-1]
    for raster in VERIFY_RASTERS:
        settings=replace(BASE,order=order,raster=raster)
        row=[]
        for u in (-1.,0.,1.):
            S=real_frequency_s(PHYS,settings,vary(p,MOD,u))
            row.append((S,polarization_metrics(S)))
        raster_results.append(dict(raster=raster,data=row))
        print(f"raster={raster}: η[-,0,+]={[round(x[1]['eta'],5) for x in row]}")
    for i in range(3):
        print(f'u={(-1,0,1)[i]} max |ΔS| from finest:',
              [round(float(np.max(abs(r['data'][i][0]-raster_results[-1]['data'][i][0]))),6)
               for r in raster_results])

## 5 · Dense physical modulation: η alone vs gap, anisotropy, power

The optimizer maximizes **span** $\max_u\eta-\min_u\eta$ over a few points. Here we assess the full interval with intermediate points, report derivatives in physical units and explicitly **do not optimize** the gap, anisotropy or power. Discontinuous eigenvalue label switches are handled by continuation, not by sorting independently at each point.

In [ ]:
U=np.linspace(-1,1,25)
curve=[]
if RUN_MODULATION_SWEEP:
    for i,u in enumerate(U):
        S=real_frequency_s(PHYS,BASE,vary(p,MOD,float(u)))
        curve.append(dict(u=float(u),value=GEOM[MOD.parameter]+MOD.amplitude*u,
                          S=S,**polarization_metrics(S)))
        if (i+1)%5==0:print(f'Modulation {i+1}/{len(U)} · η={curve[-1]["eta"]:.4f}')
    coords=np.array([r['value'] for r in curve])
    fig,axs=plt.subplots(2,3,figsize=(14,7))
    for ax,key,label in zip(axs.flat,
            ('eta','gap','anisotropy','nonnormality','transmitted_x','transmitted_y'),
            ('η','Normalized eigenvalue gap','Normalized non-scalar strength',
             'Normalized nonnormality','Transmission for x input','Transmission for y input')):
        ax.plot(coords,[r[key] for r in curve],'-o',markersize=3)
        ax.set(xlabel=MOD.parameter,ylabel=label);ax.grid(alpha=.2)
    plt.tight_layout();plt.show()
    values=np.array([r['eta'] for r in curve])
    grad=np.gradient(values,coords)
    print(f'Measured η range: {values.min():.4f} to {values.max():.4f}; span {np.ptp(values):.4f}')
    print('Maximum |dη/dp|:',np.max(abs(grad)),
          'at parameter:',coords[np.argmax(abs(grad))])
    print('Numerically undefined points:',sum(not r['defined'] for r in curve))

## 5b · All six parameter directions (different units, don't compare raw slopes)

These are independent finite differences at the optimized center. The *physical modulation range* and *fabrication uncertainty* must be considered to compare controllability.

In [ ]:
sensitivities=[]
PHYSICAL_STEPS={'length1':.008,'width1':.004,'length2':.008,
                'width2':.004,'bend_deg':4.,'junction_frac':.035}
if RUN_PARAMETER_SENSITIVITY:
    for key,step in PHYSICAL_STEPS.items():
        outs=[]
        for sign in (-1,+1):
            pp=dict(p);pp[key]=pp[key]+float(sign*step)
            S=real_frequency_s(PHYS,BASE,pp)
            outs.append(polarization_metrics(S))
        slope=(outs[1]['eta']-outs[0]['eta'])/(2*step)
        sensitivities.append((key,slope,outs[0]['eta'],outs[1]['eta']))
        print(f'{key:15s} dη/dp={slope:+.4g}, η(-,+)=({outs[0]["eta"]:.3f},{outs[1]["eta"]:.3f})')

## 6 · Eigenvalue and eigenpolarization trajectories (Poincaré sphere)

Track labels continuously in the actual Jones eigenspace. Right Stokes states live on the Poincaré sphere; reciprocal-left dual vectors are not generally the Hermitian adjoints of right eigenvectors. A large η alone does not determine the optical geometric phase.

In [ ]:
tracked=None
if curve:
    tracked=tracked_modes([r['S'][2:,:2] for r in curve])
    from engine_twofin_eta import stokes
    fig=plt.figure(figsize=(14,5))
    ax=fig.add_subplot(121,projection='3d')
    a=np.linspace(0,2*np.pi,35); b=np.linspace(0,np.pi,17)
    x=np.outer(np.cos(a),np.sin(b)); y=np.outer(np.sin(a),np.sin(b)); z=np.outer(np.ones_like(a),np.cos(b))
    ax.plot_wireframe(x,y,z,alpha=.09,color='gray',linewidth=.45)
    for branch in range(2):
        xyz=np.array([stokes(M[:,branch]) for M in tracked['R']])
        ax.plot(xyz[:,0],xyz[:,1],xyz[:,2],'-o',markersize=2.5,label=f'eigenstate {branch}')
        left_kets=tracked['L_bras'][:,branch,:].conj()
        left_kets=left_kets/np.linalg.norm(left_kets,axis=1,keepdims=True)
        left_xyz=np.array([stokes(v) for v in left_kets])
        ax.plot(left_xyz[:,0],left_xyz[:,1],left_xyz[:,2],'--',lw=1.1,
                label=f'left dual {branch}')
    ax.set(xlabel='S₂',ylabel='S₃',zlabel='S₁',title='Right eigenpolarizations')
    ax.set_box_aspect((1,1,1));ax.legend()
    ax=fig.add_subplot(122)
    for branch in range(2):
        vals=tracked['eigenvalues'][:,branch]
        ax.plot(vals.real,vals.imag,'-o',markersize=3,label=f'λ{branch}')
    ax.set(xlabel='Re eigenvalue',ylabel='Im eigenvalue',title='Tracked T21 eigenvalues',aspect='equal')
    ax.legend();ax.grid(alpha=.2)
    plt.tight_layout();plt.show()
    print('minimum normalized gap:',tracked['gap'].min())
    print('η range from tracked modes:',tracked['eta'].min(),tracked['eta'].max())

## 7 · Biorthogonal Wilson loops in $(u,\theta)$ — **virtual parameter transport**

For a closed *sequence* of right/reciprocal-left eigenvectors, $W_n=\prod_j\langle L_n(p_j)|R_n(p_{j+1})\rangle$ and $\gamma_n=i\log W_n$ under the paper's convention. We compute both directions, an orthogonalized-basis control, and optionally refine the discretization. **None of this means that light traverses adjacent meta-atoms in this order.** Large $\operatorname{Im}\gamma$ shared by both branches may be finite-step projection loss; inspect its behavior under refinement.

A winding path can exchange eigenbranches. If tracking/closing indicates a branch exchange, an Abelian one-band Wilson phase is not interpretable; use a non-Abelian path treatment instead.

In [ ]:
virtual_loops={}
if RUN_WILSON:
    for n in ([7,13,25] if RUN_WILSON_REFINEMENT else [7]):
        path=rectangular_parameter_loop(n_edge=n,umin=-.65,umax=.65,theta0=0,theta1=90)
        matrices=[]
        for i,(u,theta) in enumerate(path):
            S=real_frequency_s(PHYS,BASE,vary(p,MOD,float(u))) if theta==0 else None
            if S is None:
                with torch.no_grad():
                    S=solve_torch(vary(p,MOD,float(u)),PHYS,BASE,
                                  theta_deg=float(theta),hard=True).cpu().numpy()
            matrices.append(S[2:,:2])
            if (i+1)%12==0:print(f'Wilson n={n}: {i+1}/{len(path)} points')
        Ts=np.array(matrices)
        fw=wilson_cycle(Ts)
        rev=wilson_cycle(Ts[::-1])
        orth=wilson_cycle(Ts,orthogonal_control=True)
        virtual_loops[n]=dict(path=path,forward=fw,reverse=rev,orth=orth)
        print(f'cycle n={n}, samples={len(path)}:')
        print('  gamma forward:',fw['gamma'],' |W|',abs(fw['W']))
        print('  gamma reverse:',rev['gamma'],' |W|',abs(rev['W']))
        print('  gamma orth control:',orth['gamma'],' |W|',abs(orth['W']))
        print('  min eigenvalue gap:',np.min(fw['gap']))
        print('  branch exchange?',fw['exchange_warning'])
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    representative=virtual_loops[min(virtual_loops)]
    axs[0].plot(*representative['path'].T,'-o',ms=2)
    axs[0].set(xlabel='modulation u',ylabel='orientation θ (deg)',title='Virtual closed parameter path')
    axs[1].plot(representative['forward']['eta'],label='η')
    axs[1].plot(representative['forward']['gap'],label='eigenvalue gap')
    axs[1].set(xlabel='path index',title='Eigenvector condition along virtual loop')
    axs[1].legend();plt.tight_layout();plt.show()

## 7b · Analytic non-Hermitian loop benchmark

Independently exercise the Wilson code on $K(\phi,\eta)=\cos\phi\,\sigma_x+\sin\phi\,\sigma_y+i\eta\sigma_z$ for $\eta=0.6$, including the Hermitian $\eta=0$ control. For a dense closed sweep $\phi:0\rightarrow2\pi$ the ideal complex geometric contribution has opposite imaginary parts in the two eigenchannels; projection-induced imaginary terms in the orthogonal control should vanish as sampling is refined. This validates the *formula*, not the real metasurface.

In [ ]:
def analytic_K(phi,eta):
    return np.array([[1j*eta,np.exp(-1j*phi)],
                     [np.exp(1j*phi),-1j*eta]],complex)
for N in [16,64,256]:
    for eta in [0.,.6]:
        mats=[analytic_K(t,eta) for t in np.linspace(0,2*np.pi,N,endpoint=False)]
        W=wilson_cycle(mats)
        print(f'Analytic η={eta:.1f}, N={N:3d}: gamma={np.round(W["gamma"],4)}, '
              f'|W|={np.round(abs(W["W"]),5)}')

## 8 · Independent $(\eta,\theta)$ control and circular-conversion PB phase

An ideal rotationally covariant Jones element has orientation-independent $\eta$ and $\arg t_{R\leftarrow L}\sim\pm2\theta$; the square-lattice environment may violate continuous rotational covariance. Measure **full RCWA responses** rather than applying a Jones rotation to the central result and assuming it is exact. The phase of a near-zero conversion coefficient is undefined.

In [ ]:
PB={}
U_PB=np.linspace(-1,1,5)
THETA_PB=np.linspace(0,180,13)
if RUN_PB_LIBRARY:
    for i,u in enumerate(U_PB):
        for j,theta in enumerate(THETA_PB):
            with torch.no_grad():
                S=solve_torch(vary(p,MOD,float(u)),PHYS,BASE,
                              theta_deg=float(theta),hard=True).cpu().numpy()
            PB[i,j]=dict(S=S,**polarization_metrics(S))
        print(f'PB row {i+1}/{len(U_PB)} complete')
    Et=np.array([[PB[i,j]['eta'] for j in range(len(THETA_PB))] for i in range(len(U_PB))])
    Amp=np.array([[abs(PB[i,j]['t_R_from_L']) for j in range(len(THETA_PB))] for i in range(len(U_PB))])
    Ph=np.array([[np.angle(PB[i,j]['t_R_from_L']) for j in range(len(THETA_PB))] for i in range(len(U_PB))])
    fig,axs=plt.subplots(1,3,figsize=(15,4))
    for ax,Z,title in zip(axs,(Et,Amp**2,Ph),('η(u,θ)','|t(R←L)|²','arg t(R←L)')):
        im=ax.imshow(Z,origin='lower',aspect='auto',extent=(0,180,-1,1))
        ax.set(xlabel='θ (degrees)',ylabel='u',title=title)
        fig.colorbar(im,ax=ax)
    plt.tight_layout();plt.show()
    fig,axs=plt.subplots(1,2,figsize=(11,3.5))
    for i,u in enumerate(U_PB):
        angle=np.unwrap(Ph[i]-Ph[i,0])
        axs[0].plot(THETA_PB,np.degrees(angle),label=f'u={u:+.1f}')
        axs[1].plot(THETA_PB,Et[i],label=f'u={u:+.1f}')
        phi_ref=2*np.deg2rad(THETA_PB)
        residual=min(np.sqrt(np.mean(np.angle(np.exp(1j*(angle-phi_ref)))**2)),
                     np.sqrt(np.mean(np.angle(np.exp(1j*(angle+phi_ref)))**2)))
        print(f'u={u:+.1f} ηθ-span={np.ptp(Et[i]):.3f}, '
              f'min circular conversion={Amp[i].min():.3g}, '
              f'best ±2θ residual RMS={residual:.3f} rad')
    axs[0].plot(THETA_PB,2*THETA_PB,'k--',lw=1,label='+2θ')
    axs[0].plot(THETA_PB,-2*THETA_PB,'k:',lw=1,label='-2θ')
    axs[0].set(xlabel='θ (degrees)',ylabel='Unwrapped conversion phase (deg)',title='PB winding')
    axs[1].set(xlabel='θ (degrees)',ylabel='η',title='Rotation dependence of η')
    for ax in axs:ax.legend(fontsize=7);ax.grid(alpha=.2)
    plt.tight_layout();plt.show()

## 9 · Real-frequency complex response: amplitudes, eigenvalues, power, $eta$

Sweep actual wavelength at fixed *physical* geometry and constant nondispersive refractive indices. This isolates scattering resonances from geometry modulation. If measured materials are dispersive, replace these constant indices with a causal dispersion model **before** interpreting pole locations. The 4×4 matrix is kept intact; reflection and transmission are both plotted.

In [ ]:
spectra=[]
WAVELENGTHS=np.linspace(.57,.71,21)
SPECTRAL_OPTS=replace(BASE,order=6,raster=192)
if RUN_REAL_SPECTRUM:
    for i,wl in enumerate(WAVELENGTHS):
        S=real_frequency_s(replace(PHYS,wavelength=float(wl)),SPECTRAL_OPTS,p)
        spectra.append(dict(wavelength=float(wl),S=S,**polarization_metrics(S)))
        if (i+1)%5==0:print(f'Wavelength {i+1}/{len(WAVELENGTHS)} · {wl*1e3:.1f} nm')
    fig,axs=plt.subplots(2,3,figsize=(15,8))
    for ax,name in zip(axs.flat,('eta','gap','nonnormality','transmitted_x','transmitted_y','reflected_y')):
        ax.plot(1e3*np.array([r['wavelength'] for r in spectra]),
                [r[name] for r in spectra],'-o',ms=2.5)
        ax.set(xlabel='Wavelength (nm)',ylabel=name,title=name);ax.grid(alpha=.2)
    plt.tight_layout();plt.show()
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    vals=np.array([r['eigenvalues'] for r in spectra])
    for j in range(2):axs[0].plot(vals[:,j].real,vals[:,j].imag,'-o',ms=2.5,label=f'eig{j} (untracked)')
    axs[0].set(xlabel='Re eigenvalue',ylabel='Im eigenvalue',title='T21 eigenvalues (untracked)',aspect='equal');axs[0].legend()
    xx=np.array([r['S'][2,0] for r in spectra])
    axs[1].plot([r['wavelength']*1000 for r in spectra],abs(xx),label='|t_xx|')
    axs[1].plot([r['wavelength']*1000 for r in spectra],np.unwrap(np.angle(xx)),label='unwrap arg t_xx')
    axs[1].set(xlabel='Wavelength (nm)',title='Complex scattering coefficient');axs[1].legend();axs[1].grid(alpha=.2)
    plt.tight_layout();plt.show()

## 10 · Complex-frequency scattering: exploratory pole/zero structure **[optional]**

We evaluate $S(f)$ at **complex** $f=c^{-1}\omega/2\pi$ (units µm$^{-1}$, $c=1$): $\operatorname{Im} f<0$ is the lower half-plane for $e^{-i\omega t}$. This is analytic continuation of the frequency-domain boundary-value problem, **not** a physical exponentially growing incident laser.

**Crucial implementation:** real-frequency TORCWA `power_norm=True` uses real/imaginary $k_z$ tests and real-part factors. For nonreal frequency that is **not an analytic scattering normalization**. The helper instead uses `power_norm=False` and a frequency-independent, normal-incidence port-index scaling. The first test compares that matrix with the established real-frequency solver and **raises** on disagreement; no pole figures are produced in that case. We also require $n_{\mathrm{top}}, n_{\mathrm{bottom}}$ constant in this exploratory continuation.

Two-dimensional maps show $\log_{10}\sigma_{\max}(S)$, $\log_{10}|\det S|$, and a selected complex transmission entry's phase. Rational fits are **proxies**, not identified physical quasinormal-mode poles: verify frequency window, contour, Fourier order, numerical branches and residuals independently. Near Rayleigh branch cuts, polynomial/rational approximations can mislead.

In [ ]:
complex_samples=None
complex_fit=None
COMPLEX_ORDER=3
COMPLEX_RASTER=128
COMPLEX_REAL_POINTS=13
COMPLEX_IMAG_POINTS=9
COMPLEX_FRACTION=0.08  # fractional half-width near f0
if RUN_COMPLEX_FREQUENCY:
    complex_opts=RCWA(order=COMPLEX_ORDER,raster=COMPLEX_RASTER,
                      precision='complex128',device=DEVICE)
    f0=1/PHYS.wavelength
    S_real=real_frequency_s(PHYS,complex_opts,p)
    S_cont=solve_complex_s(p,PHYS,complex_opts,freq=f0)
    real_axis_error=np.linalg.norm(S_cont-S_real)/max(np.linalg.norm(S_real),1e-20)
    print('Complex solver / real-axis solver relative mismatch:',real_axis_error)
    if not np.isfinite(real_axis_error) or real_axis_error>2e-4:
        raise RuntimeError('Complex-frequency normalization/interface mismatch. '
                           'Do not interpret its pole/zero map until this is resolved.')
    fr=np.linspace(f0*(1-COMPLEX_FRACTION),f0*(1+COMPLEX_FRACTION),COMPLEX_REAL_POINTS)
    fi=np.linspace(-f0*COMPLEX_FRACTION,0.,COMPLEX_IMAG_POINTS)
    matrix=np.empty((len(fi),len(fr),4,4),complex)
    valid=np.zeros(matrix.shape[:2],bool)
    for i,y in enumerate(fi):
        for j,x in enumerate(fr):
            try:
                sol=solve_complex_s(p,PHYS,complex_opts,freq=x+1j*y)
                matrix[i,j]=sol
                valid[i,j]=np.isfinite(sol).all()
            except (RuntimeError,ValueError,np.linalg.LinAlgError) as exc:
                matrix[i,j]=np.nan+1j*np.nan
                print(f'Complex ({i},{j}) failed:',str(exc)[:100])
        print(f'Complex-frequency grid row {i+1}/{len(fi)} · finite samples {valid[i].sum()}')
    complex_samples={'real':fr,'imag':fi,'S':matrix,'valid':valid,
                     'real_axis_error':real_axis_error,'order':COMPLEX_ORDER}
    largest=np.full(valid.shape,np.nan);ldet=np.full(valid.shape,np.nan)
    phase=np.angle(matrix[:,:,2,0])
    for ij in np.argwhere(valid):
        i,j=ij
        singular=np.linalg.svd(matrix[i,j],compute_uv=False)
        largest[i,j]=np.log10(max(singular[0],1e-30))
        ldet[i,j]=np.log10(max(abs(np.linalg.det(matrix[i,j])),1e-30))
    fig,axs=plt.subplots(1,4,figsize=(18,4))
    log_t=np.log10(np.maximum(abs(matrix[:,:,2,0]),1e-30))
    for ax,Z,title in zip(axs,(largest,ldet,log_t,phase),
                          ('log₁₀ max σ(S)','log₁₀ |det S|','log₁₀ |S[2,0]|','arg S[2,0]')):
        img=ax.imshow(Z,origin='lower',aspect='auto',
                extent=(fr[0],fr[-1],fi[0],fi[-1]))
        ax.set(xlabel='Re f (1/µm)',ylabel='Im f (1/µm)',title=title)
        fig.colorbar(img,ax=ax)
    plt.tight_layout();plt.show()

    # Hold out every fourth complex sample to audit a two-pole rational fit.
    grid=(fr[None,:]+1j*fi[:,None])[valid]
    sxy=matrix[:,:,2,0][valid]
    split=np.arange(len(grid))%4!=0
    if split.sum()>12 and (~split).sum()>4:
        complex_fit=rational_fit(grid[split],sxy[split],poles=2,center=f0)
        q=complex_fit
        z=(grid[~split]-q['center'])/q['scale']
        predictions=np.polynomial.polynomial.polyval(z,q['numerator'])/                    np.polynomial.polynomial.polyval(z,q['denominator'])
        heldout_err=np.linalg.norm(predictions-sxy[~split])/max(np.linalg.norm(sxy[~split]),1e-30)
        print('Rational fit training relative RMS:',q['relative_error'])
        print('Rational fit held-out relative RMS:',heldout_err)
        print('Fit matrix condition:',q['condition'])
        print('Candidate rational poles (1/µm):',q['poles'])
        print('Candidate rational zeros (1/µm):',q['zeros'])
        print('Do not label these as Maxwell QNMs without order/window continuation checks.')
    # Independent nearby-order check at selected complex frequencies.
    higher_opts=replace(complex_opts,order=COMPLEX_ORDER+1)
    for test_f in (f0-0.025*f0-0.02j*f0, f0+0.01*f0-0.04j*f0):
        low=solve_complex_s(p,PHYS,complex_opts,freq=test_f)
        high=solve_complex_s(p,PHYS,higher_opts,freq=test_f)
        diff=np.linalg.norm(low-high)/max(np.linalg.norm(high),1e-30)
        print(f'Complex-frequency RCWA order {COMPLEX_ORDER}→{COMPLEX_ORDER+1} '
              f'at f={test_f:.5g}: relative full-S change {diff:.3g}')

## 11 · Actual depth-resolved Maxwell fields **[optional]**

Sample the unit-cell-averaged $E_x,E_y,H_x,H_y$ as functions of depth $z$ for independent $x/y$ illuminations. Since this is the **total** field (forward and reflected/backward components), its $z$-dependent two-by-two response is **not** the local forward propagation operator $K(z)$. This can falsify an assumed simple forward-only model, but cannot alone certify complex geometric phase.

In [ ]:
depth=None
if RUN_Z_FIELDS:
    depth_opts=replace(BASE,order=5,raster=192)
    depth=depth_response(p,PHYS,depth_opts,nz=17,xy_samples=12)
    z=depth['z'];Ez=depth['E_xy_response'];Hz=depth['H_xy_response']
    fig,axs=plt.subplots(1,3,figsize=(14,4))
    for input_index in range(2):
        for output_index in range(2):
            axs[0].plot(z,abs(Ez[:,output_index,input_index])**2,
                        label=f'{"xy"[output_index]} ← {"xy"[input_index]}')
    axs[0].set(xlabel='z (µm)',ylabel='|⟨E⟩|²',title='Cell-averaged total E')
    axs[0].legend(fontsize=7)
    depth_eta=[]
    for mat in Ez:
        w,V=np.linalg.eig(mat); V=V/np.linalg.norm(V,axis=0)
        depth_eta.append(abs(np.vdot(V[:,0],V[:,1])))
    axs[1].plot(z,depth_eta,'-o',ms=3)
    axs[1].set(xlabel='z (µm)',ylabel='eigenvector overlap',title='Response-map η(z), NOT local η(K)')
    e1=Ez@np.array([1.,0.]);e2=Ez@np.array([0.,1.])
    axs[2].plot(z,abs(e1[:,0])**2+abs(e1[:,1])**2,label='x launch')
    axs[2].plot(z,abs(e2[:,0])**2+abs(e2[:,1])**2,label='y launch')
    axs[2].set(xlabel='z (µm)',ylabel='|⟨E⟩|²',title='Projected field intensity')
    axs[2].legend()
    for ax in axs:ax.grid(alpha=.2)
    plt.tight_layout();plt.show()

## 12 · Spatial ordering: compare local Jones predictions with actual Maxwell arrays **[optional]**

Spatial neighbors under normal illumination are illuminated **in parallel**, not sequentially. To probe interpixel coupling, simulate a $3\times3$ periodic supercell with the same histogram of $(u,\theta)$ but different spatial arrangement. Compare *all propagating diffraction-order* power budgets, not a single specular Jones block. Reordering changes diffraction even without geometric transport: an order-dependent diffraction pattern **is not sufficient evidence** of Wilson holonomy.

In [ ]:
array_results={}
if RUN_SUPERCELL_ORDER_TEST:
    nx=3
    uarr=np.repeat(np.linspace(-.7,.7,nx)[:,None],nx,axis=1)
    tarr=np.repeat(np.linspace(0,90,nx)[None,:],nx,axis=0)
    cases={'joint':(uarr,tarr),
           'spatially_reversed':(uarr[::-1,::-1],tarr[::-1,::-1]),
           'only_u':(uarr,np.zeros_like(tarr)),
           'only_theta':(np.zeros_like(uarr),tarr),
           'uniform':(np.zeros_like(uarr),np.zeros_like(tarr))}
    # Each pixel must have enough raster samples; increasing Fourier order is costly.
    sc_opts=RCWA(order=9,raster=384,device=DEVICE,precision='complex128')
    for label,(us,ts) in cases.items():
        with torch.no_grad():Sspec,orders=supercell_solve(us,ts,p,MOD,PHYS,sc_opts)
        # The helper supplies zero-order input power-normalized channel blocks for
        # every diffraction order propagating in BOTH exterior media.
        trans=sum((abs(S[2:,:2])**2).sum(axis=0) for S in orders.values())
        refl=sum((abs(S[:2,:2])**2).sum(axis=0) for S in orders.values())
        power=trans+refl
        array_results[label]=dict(Sspec=Sspec,trans=trans,refl=refl,
                                  power=power,order_count=len(orders))
        print(label,'orders',len(orders),'T x/y',np.round(trans,4),
              'R x/y',np.round(refl,4),'total x/y',np.round(power,4))
        if np.max(abs(power-1))>.02:
            print('WARNING: power conservation/retained-order budget not converged.')

In [ ]:
finite_fields=None
if RUN_FINITE_ARRAY:
    from engine_twofin_finite import FiniteSettings,run_finite_meep,field_diagnostics
    n=3
    uarr=np.repeat(np.linspace(-.65,.65,n)[:,None],n,axis=1)
    tarr=np.repeat(np.linspace(0,90,n)[None,:],n,axis=0)
    finite_fields=run_finite_meep(uarr,tarr,p,MOD,PHYS,
                        opts=FiniteSettings(resolution=60,run_time=200.))
    print('Finite Meep field diagnostics (not local K or a measured Wilson loop):')
    for x in field_diagnostics(finite_fields):print(x)

## 13 · Save complete numerical evidence and interpretation gates

Save complex $S(\lambda)$, modulation traces, Wilson controls, optional complex-frequency data and the actual optimizer checkpoint reference. **No generated plot constitutes physical EP/complex-holonomy proof without convergence and a justified evolution operator.**

In [ ]:
manifest={'checkpoint':str(saved['path']),'checkpoint_objective':saved['tag'],
          'physics':asdict(PHYS),'modulation':asdict(MOD),'geometry':GEOM,
          'base_rcwa':asdict(BASE),'orders':VERIFY_ORDERS,
          'interpretation':'T21 eta is specular eigenpolarization overlap. Wilson parameter loops are virtual, not ordered optical transport.'}
arrays={'manifest':json.dumps(manifest), 'S_center':S0,
        'raw':RAW,'parameters':np.array([GEOM[k] for k in PARAM_KEYS])}
if curve:
    arrays.update(modulation_u=U,modulation_values=np.array([r['value'] for r in curve]),
        modulation_S=np.array([r['S'] for r in curve]),
        modulation_eta=np.array([r['eta'] for r in curve]),
        modulation_gap=np.array([r['gap'] for r in curve]))
if spectra:
    arrays.update(spectral_wavelength=np.array([r['wavelength'] for r in spectra]),
        spectral_S=np.array([r['S'] for r in spectra]))
if PB:
    arrays.update(pb_u=U_PB,pb_theta=THETA_PB,
        pb_S=np.array([[PB[i,j]['S'] for j in range(len(THETA_PB))]
                       for i in range(len(U_PB))]))
if virtual_loops:
    key=min(virtual_loops); q=virtual_loops[key]
    arrays.update(virtual_path=q['path'],virtual_W=q['forward']['W'],
                  virtual_gamma=q['forward']['gamma'],
                  virtual_orth_W=q['orth']['W'])
if complex_samples:
    arrays.update(complex_real=complex_samples['real'],
                  complex_imag=complex_samples['imag'],
                  complex_S=complex_samples['S'],complex_valid=complex_samples['valid'])
if depth:
    arrays.update(depth_z=depth['z'],depth_E=depth['E_xy_response'],
                  depth_H=depth['H_xy_response'])
name=OUT/'twofin_eta_characterization.npz'
np.savez_compressed(name,**arrays)
print('Saved:',name.resolve())
print('η center:',round(M0['eta'],4),'gap:',round(M0['gap'],4),
      'anisotropy:',round(M0['anisotropy'],4))
print('Virtual Wilson product is not a measured optical phase in the one-layer array.')
print('Actual evidence for ordered transport requires a propagating in-plane path, '
      'validated local evolution/transport operator, and dynamical/resonant controls.')